In [2]:
# [CELDA 1] - Importaciones y configuración de variables MLOps
%pip install mlflow ultralytics boto3

import os
import mlflow
import mlflow.pytorch
from ultralytics import YOLO

# Configuramos la URL local del servidor de MLflow y las credenciales de MinIO
os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5001"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"

# Definimos el experimento
EXPERIMENT_NAME = "Deteccion_Humos_Aluar"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"🔗 Conectado a MLflow Tracking en: {os.environ['MLFLOW_TRACKING_URI']}")

# ---

# [CELDA 2] - Configuración de hiperparámetros
CONFIG = {
    "architecture": "YOLOv8n",
    "model_pretrained": "yolov8n.pt",
    "epochs": 15,
    "img_size": 640,
    "batch_size": 16,
    "data_yaml": os.path.abspath("../dataset/data.yaml")
}

# Verificamos que el archivo data.yaml exista
if not os.path.exists(CONFIG["data_yaml"]):
    raise FileNotFoundError(f"No se encontró el archivo {CONFIG['data_yaml']}. Ejecutá primero el notebook 01_download_dataset.ipynb.")

# ---

# [CELDA 3] - Entrenamiento del modelo e integración con MLflow
run_name = f"yolov8n_smoke_run"

with mlflow.start_run(run_name=run_name) as run:
    print(f"🚀 Iniciando Run en MLflow con ID: {run.info.run_id}")
    
    # 1. Registrar Hiperparámetros
    mlflow.log_params(CONFIG)
    
    # 2. Cargar modelo base
    model = YOLO(CONFIG["model_pretrained"])
    
    # 3. Entrenar
    print("⏳ Entrenando el modelo...")
    results = model.train(
        data=CONFIG["data_yaml"],
        epochs=CONFIG["epochs"],
        imgsz=CONFIG["img_size"],
        batch=CONFIG["batch_size"],
        project="../runs",
        name="train_exp"
    )
    
    # 4. Evaluar métricas en el conjunto de validación
    metrics = model.val()
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    
    print(f"📊 Métricas obtenidas -> mAP50: {map50:.4f} | mAP50-95: {map50_95:.4f}")
    
    # 5. Registrar métricas en MLflow
    mlflow.log_metrics({
        "mAP50": map50,
        "mAP50_95": map50_95
    })
    
    # 6. Guardar pesos del mejor modelo como Artefacto
    best_weights = os.path.abspath("../runs/train_exp/weights/best.pt")
    if os.path.exists(best_weights):
        print(f"📦 Subiendo {best_weights} a MinIO/MLflow...")
        mlflow.log_artifact(best_weights, artifact_path="model_weights")
    
    print("🎉 ¡Entrenamiento y logueo en MLflow/MinIO completado exitosamente!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 5.4 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 6.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 6.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 7.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 7.7 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 9.7 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 8.4 MB/s  0:00:00 eta 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 8.5 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 9.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.3 MB/s  0:0

2026/07/31 00:36:59 INFO mlflow.tracking.fluent: Experiment with name 'Deteccion_Humos_Aluar' does not exist. Creating a new experiment.


🔗 Conectado a MLflow Tracking en: http://localhost:5001


FileNotFoundError: No se encontró el archivo /Volumes/VIRTUAL WIN/MAESTRIAS/ESPECIALIZACIÓN EN INTELIGENCIA ARTIFICIAL/CURSO 12 - TALLER INTEGRADOR A/MLOPS_deteccion_humos/dataset/data.yaml. Ejecutá primero el notebook 01_download_dataset.ipynb.